# External re-test of the frozen joint graders on DDR and IDRiD, then uncertainty — record §96

**Run only after both joint arms are complete (six runs) and the user has authorised it.** Evaluation only: nothing is
trained, tuned or selected. This is a documented **re-use** of both test sets (DDR: second grading use; IDRiD: fourth).

- **Main contrasts, equal standing — neither joint arm is the pre-declared final model:** QWK(J) − QWK(P) and
  QWK(J-crop) − QWK(P), each the mean of the three seed-labelled pairs, on each dataset's primary population (DDR 3,540;
  IDRiD 100). Each is read on its own: supported only if its paired 95 % interval is wholly above zero; a non-positive seed
  difference is flagged. Intervals are not adjusted for the two contrasts.
- **Secondary, descriptive:** J − P-EP, J-crop − P-EP, J-crop − J; per-grade and per-cut metrics; calibration; seed ensemble.
- A choice between the two arms made from these results is post hoc, and the chosen arm's estimate is then optimistic.
- P and P-EP are **not run again**: their stored per-image tables from the earlier runs are reused after their hashes are checked.

Order of the cells — it cannot be changed:
1. Setup. 2. Switches. 3. Protocol: the frozen file `ddr_idrid_retest_protocol_v1.json` from the repository, re-verified against Drive.
4. Parity: every joint model must reproduce its own APTOS validation table from the raw images. If it fails, stop.
5. DDR re-test, once. 6. IDRiD re-test, once. 7. Uncertainty of the three-seed ensemble (no GPU needed). 8. Summary.

**Limits that travel with the result:** both sets have been used before and the earlier results were known when joint
training was chosen; P's batch size was 2 and the joint models' 16; on DDR file-name family is tied to grade and there are
no patient identifiers; IDRiD's primary set has 100 images; nothing here establishes clinical utility.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. 
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad
import shutil, tarfile
import ddr_grading_data as dd, ddr_grading_eval as ev
import joint_train as jt, final_external_eval as fx, uncertainty_eval as ue, gradcam, demo_pipeline as demo
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] SWITCHES -- everything is OFF. A run needs its flag AND its token. ====
RUN_PARITY    = False    # APTOS parity of the six joint models from raw images (about 20 min)
CONFIRM_DDR   = None     # the DDR token printed in cell 3 -- the DDR re-test runs once
CONFIRM_IDRID = None     # the IDRiD token printed in cell 3 -- the IDRiD re-test runs once
TECHNICAL_RERUN_REASON = None
print({"RUN_PARITY": RUN_PARITY, "DDR": CONFIRM_DDR is not None, "IDRiD": CONFIRM_IDRID is not None})

In [ ]:
# ==== [3] PROTOCOL (no model, no external image) ====
DRIVE_ROOT = colab_config.DRIVE.project_root
EXP = colab_config.DRIVE.experiments_root
PROTOCOL_PATH = fx.PROTOCOL_PATH            # the frozen protocol committed in the repository (never rewritten here)
print("joint runs:", {a: {s: jt.state(jt.run_dir_for(EXP, s, a)) for s in jt.SEEDS} for a in jt.ARMS})
if os.path.exists(PROTOCOL_PATH):
    PROTOCOL, PROTOCOL_SHA, DDR_PROTOCOL, PARENT = fx.load_protocol(PROTOCOL_PATH)
    FILES = fx.verify_files(PROTOCOL, DRIVE_ROOT, PARENT)                                  # six checkpoints and the stored reference tables, by hash
    print("protocol", PROTOCOL_SHA, "| commit", REPO_COMMIT)
    for _name, _entry in PROTOCOL["checkpoints"].items():
        print(f"  {_name:<12} {_entry['sha256']}  epoch index {_entry['best_epoch_index']}")
    print("main (equal standing):", PROTOCOL["comparisons"]["main"], "| descriptive:", PROTOCOL["comparisons"]["secondary_descriptive"])
    print("tokens:", PROTOCOL["confirmation_tokens"])
else:
    raise RuntimeError("the frozen protocol file is missing from the repository")

In [ ]:
# ==== [4] PARITY -- must pass before any external image is read ====
GATE = None
if RUN_PARITY:
    GATE = fx.parity(DRIVE_ROOT, posixpath.join(colab_config.APTOS2019_RAW_DIR, "train_images"),
                     posixpath.join(EXP, "FinalExternal", f"parity_{PROTOCOL_SHA[:12]}"), PROTOCOL_PATH)
    print(json.dumps({"PASS": GATE["PASS"], "failures": GATE["failures"], "checks": GATE["checks"]}, indent=1))
    assert GATE["PASS"], "PARITY FAILED -- stop here. Do not run the external cells. Report the failures."
else:
    print("parity not run (RUN_PARITY = False)")

In [ ]:
# ==== [5] DDR RE-TEST, ONCE (six joint models on the 3,725 listed images; about 1.5 h) ====
if CONFIRM_DDR is not None:
    assert GATE is not None and GATE["PASS"], "run the parity cell first, in this runtime"
    SOURCE = posixpath.join(colab_config.DATASET_ROOT, "DDR", "grading_test_v1")
    LOCAL = "/content/ddr_grading_test_v1"
    IMAGES, AUDIT = posixpath.join(LOCAL, "images"), posixpath.join(LOCAL, "audit")
    if not os.path.isdir(IMAGES) or len(os.listdir(IMAGES)) != DDR_PROTOCOL["populations"]["broader"]["n"]:
        os.makedirs(IMAGES, exist_ok=True)
        _tar = posixpath.join(LOCAL, "images.tar")
        shutil.copyfile(posixpath.join(SOURCE, "ddr_grading_test_3725.tar"), _tar)
        with tarfile.open(_tar) as _archive:
            assert all(m.isfile() and "/" not in m.name and m.name.endswith(".jpg") for m in _archive.getmembers()), "unexpected archive member"
            _archive.extractall(IMAGES)
        os.remove(_tar)
    shutil.copytree(posixpath.join(SOURCE, "audit"), AUDIT, dirs_exist_ok=True)
    DDR_RESULT = fx.run_ddr(DRIVE_ROOT, IMAGES, AUDIT, GATE, confirm=CONFIRM_DDR, technical_rerun_reason=TECHNICAL_RERUN_REASON,
                            protocol_path=PROTOCOL_PATH)
else:
    print("DDR re-test not run (CONFIRM_DDR is None)")

In [ ]:
# ==== [6] IDRiD RE-TEST, ONCE (six joint models on the 103 images; about 15 min) ====
if CONFIRM_IDRID is not None:
    assert GATE is not None and GATE["PASS"], "run the parity cell first, in this runtime"
    IDRID_RESULT = fx.run_idrid(DRIVE_ROOT, colab_config.IDRID_GRADING_RAW_DIR, GATE, confirm=CONFIRM_IDRID,
                                technical_rerun_reason=TECHNICAL_RERUN_REASON, protocol_path=PROTOCOL_PATH)
else:
    print("IDRiD re-test not run (CONFIRM_IDRID is None)")

In [ ]:
# ==== [7] UNCERTAINTY of the three-seed ensembles (reads per-image tables that exist; no GPU needed) ====
def _joint_tables(arm, name):
    return {s: posixpath.join(jt.run_dir_for(EXP, s, arm), "metrics", f"per_sample_best_{name}.csv") for s in jt.SEEDS}

SETS = {f"{model}_{name}": _joint_tables(arm, name) for model, arm in fx.ARMS.items() for name in ("aptos_val", "eyepacs_val")}
for _label, _result in (("ddr", globals().get("DDR_RESULT")), ("idrid", globals().get("IDRID_RESULT"))):
    if _result is not None:                                           # the external tables, if the re-test ran in this runtime
        SETS.update({f"{model}_{_label}": {s: posixpath.join(_result["out_dir"], f"per_image_{model}_seed{s}.csv") for s in jt.SEEDS} for model in fx.ARMS})
SETS = {k: v for k, v in SETS.items() if all(os.path.exists(p) for p in v.values())}
if SETS:
    UNCERTAINTY = ue.run(SETS, posixpath.join(EXP, "FinalExternal", "uncertainty"))
    for _name, _r in UNCERTAINTY.items():
        _e = _r["scores"]["entropy"]
        print(f"{_name}: ensemble QWK {_r['ensemble_qwk']:.4f} | seeds disagree on {100 * _r['seeds_disagree_on_grade']:.1f} % | "
              f"error-detection AUROC (entropy) {_e['error_detection_auroc']:.3f} | QWK at 80 % coverage {[x for x in _e['selective'] if x['coverage'] == 0.8][0]['qwk']:.4f}")
else:
    print("no complete set of three seed tables yet")

In [ ]:
# ==== [8] SUMMARY of the re-test (reads what cells 5 and 6 wrote) ====
for _label, _result, _key in (("DDR", globals().get("DDR_RESULT"), "primary_3540"), ("IDRiD", globals().get("IDRID_RESULT"), "primary_100")):
    if _result is None:
        print(_label, "not run in this runtime")
        continue
    _r = _result["results"][_key]
    print(f"== {_label}: {_r['n']} images {_r['grade_counts']} | output {_result['out_dir']}")
    for _name, _d in _r["contrasts_qwk"].items():
        print(f"  {_name:<14} {_d['mean']:+.4f} [{_d['ci_low']:+.4f}, {_d['ci_high']:+.4f}] per seed { {k: round(v, 4) for k, v in _d['per_seed'].items()} }"
              f" | {'MAIN' if _name in fx.MAIN else 'descriptive'} | {_d['reading'].split(':')[0]}{' | SEED-LEVEL INCONSISTENCY' if _d['seed_level_inconsistency'] else ''}")
    print("  seed-ensemble QWK", {k: round(v, 4) for k, v in _r["seed_ensemble_qwk"].items()})